# Recuperação de Fusão na Pesquisa de Documentos
## Visão Geral
Este código implementa um sistema Fusion Retrieval que combina busca de similaridade baseada em vetores com recuperação BM25 baseada em palavras-chave. A abordagem visa aproveitar os pontos fortes de ambos os métodos para melhorar a qualidade global e a relevância da recuperação de documentos.

## Motivação
Métodos tradicionais de recuperação muitas vezes dependem de compreensão semântica (baseada em vetor) ou correspondência de palavras-chave (BM25). Cada abordagem tem seus pontos fortes e fracos. A recuperação de fusão visa combinar esses métodos para criar um sistema de recuperação mais robusto e preciso que possa lidar com uma gama mais ampla de consultas de forma eficaz.

## Componentes-chave

1. Processamento e chunks de texto PDF
2. Criação de armazenamento de vetor usando incorporações FAISS e OpenAI
3. Criação de índice BM25 para recuperação baseada em palavras-chave
4. Fusionando BM25 e resultados de busca vetorial para melhor recuperação

## Detalhes do método
Pré-processamento de documentos

1. O PDF é carregado e dividido em chunks usando SentenceSplitter.
2. Chunks são limpos substituindo 't' por espaços e limpeza de nova linha (provavelmente abordando um problema de formatação específica).

Criação da Loja Vector

1. As embeddings da OpenAI são usadas para criar representações vetoriais dos chunks de texto.
2. Uma vector store FAISS é criada a partir desses embeddings para uma busca de similaridade eficiente.

## Criação do Índice BM25

1. Um índice BM25 é criado a partir dos mesmos chunks de texto usados para o vector store.
2. Isso permite a recuperação baseada em palavras-chave ao lado do método baseado em vetores.

## Consulta de Recuperação de Fusão

Após a criação dos dois índices Consulta Fusion Retrieval combina- os para permitir uma recuperação híbrida

## Benefícios desta abordagem
1. Melhor qualidade de recuperação: Ao combinar busca semântica e baseada em palavras-chave, o sistema pode capturar similaridade conceitual e correspondências exatas de palavras-chave.
2. Flexibilidade: O parâmetro `retriever_weights` permite ajustar o equilíbrio entre busca de vetores e palavras-chave com base em casos de uso específicos ou tipos de consulta.
3. Robustness: A abordagem combinada pode lidar eficazmente com uma gama mais vasta de consultas, atenuando as deficiências dos métodos individuais.
4. Personalizabilidade: O sistema pode ser facilmente adaptado para usar diferentes lojas de vetores ou métodos de recuperação baseados em palavras-chave.

## Conclusão
Recuperação de fusão representa uma abordagem poderosa para a pesquisa de documentos que combina os pontos fortes da compreensão semântica e correspondência de palavras-chave. Ao alavancar os métodos de recuperação baseados em vetores e BM25, ele oferece uma solução mais abrangente e flexível para tarefas de recuperação de informações. Esta abordagem tem aplicações potenciais em vários campos onde tanto semelhança conceitual quanto relevância de palavras-chave são importantes, como pesquisa acadêmica, busca legal de documentos ou motores de busca de propósito geral.

# Instalação e Importações do Pacote
A célula abaixo instala todos os pacotes necessários para executar este notebook.
.

In [ ]:
# Install required packages
!pip install faiss-cpu llama-index python-dotenv

In [ ]:
import os
import sys
from dotenv import load_dotenv
from typing import List
from llama_index.core import Settings
from llama_index.core.readers import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.schema import BaseNode, TransformComponent
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core import VectorStoreIndex
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.legacy.retrievers.bm25_retriever import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever
import faiss

# Original path append replaced for Colab compatibility
# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

# Llamaindex global settings for llm and embeddings
EMBED_DIMENSION=512
Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0.1)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small", dimensions=EMBED_DIMENSION)

### Ler os Documentos

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf


In [ ]:
path = "data/"
reader = SimpleDirectoryReader(input_dir=path, required_exts=['.pdf'])
documents = reader.load_data()
print(documents[0])

Criar a Loja Vector

In [ ]:
# Create FaisVectorStore to store embeddings
fais_index = faiss.IndexFlatL2(EMBED_DIMENSION)
vector_store = FaissVectorStore(faiss_index=fais_index)

Transformação de Limpador de Texto

In [ ]:
class TextCleaner(TransformComponent):
    """
    Transformation to be used within the ingestion pipeline.
    Cleans clutters from texts.
    """
    def __call__(self, nodes, **kwargs) -> List[BaseNode]:
        
        for node in nodes:
            node.text = node.text.replace('\t', ' ') # Replace tabs with spaces
            node.text = node.text.replace(' \n', ' ') # Replace paragprah seperator with spacaes
            
        return nodes

Pipeline de ingestão

In [ ]:
# Pipeline instantiation with: 
# node parser, custom transformer, vector store and documents
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        TextCleaner()
    ],
    vector_store=vector_store,
    documents=documents
)

# Run the pipeline to get nodes
nodes = pipeline.run()

## Retrievers

BM25 Retriever

In [ ]:
bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=2,
)

## Vector Retriever

In [ ]:
index = VectorStoreIndex(nodes)
vector_retriever = index.as_retriever(similarity_top_k=2)

Fusionando ambos os recrutas

In [ ]:
retriever = QueryFusionRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever
    ],
    retriever_weights=[
        0.6, # vector retriever weight
        0.4 # BM25 retriever weight
    ],
    num_queries=1, 
    mode='dist_based_score',
    use_async=False
)

Sobre parâmetros

1. `num_queries`: Fusão de Consulta Retriever não só combina retrievers, mas também pode gerar várias perguntas de uma determinada consulta. Este parâmetro controla quantas consultas totais serão passadas para os recuperadores. Portanto, configurá-lo para 1 desabilita a geração de consulta e o retriever final só usa a consulta inicial.
2. `mode`: Existem 4 opções para este parâmetro.
- ** reciprocal_ rerank**: Aplica classificação reciproca. (Como não há normalização, este método não é adequado para este tipo de aplicação. Beacuse retrirevers diferentes retornará escalas de pontuação)
- ** pontuação relativa**: Aplica MinMax com base nas pontuações min e max entre todos os nós. Em seguida, escalonou para ser entre 0 e 1. Finalmente, os escores são ponderados pelos recuperadores relativos baseados no `retriever_weights`.
```math
      min\_score = min(scores)
      \\ max\_score = max(scores)
      ```
- **Dist_based_score**: A única diferença em relação ao `relative_score` é que o sclaing MinMax é baseado na média e std dos escores. Escalar e pesar é o mesmo.
```math
       min\_score = mean\_score - 3 * std\_dev
      \\ max\_score = mean\_score + 3 * std\_dev
      ```
- ** Simples**: Este método é simplesmente leva a pontuação máxima de cada chunk.

### Usar exemplo de caso

In [ ]:
# Query
query = "What are the impacts of climate change on the environment?"

# Perform fusion retrieval
response = retriever.retrieve(query)

## Imprimir os nós finais com pontuações

In [ ]:
for node in response:
    print(f"Node Score: {node.score:.2}")
    print(f"Node Content: {node.text}")
    print("-"*100)

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--fusion-retrieval-with-llamaindex)